# 6. Modeling Phase: Decision Tree — Explaining Customer Behavior
## Supervised Interpretation of Unsupervised Segments (CRISP-DM)
K-Means identified *which* cluster each customer belongs to. The Decision Tree answers the more important question: **why?**

By training a classifier where the target is the cluster label and the features are the original business metrics, we extract human-readable if/then rules that explain segmentation logic in plain business language.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded.")


### 6.1 Load Cluster-Labeled Customer Profiles


In [ ]:
data_dir = 'data'
df = pd.read_csv(os.path.join(data_dir, 'customer_clusters.csv'))

print(f"Dataset loaded: {df.shape[0]} customers, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nCluster distribution:\n{df['Cluster'].value_counts().sort_index()}")
df.head()


### 6.2 Feature Engineering for Supervised Learning
We separate the dataset into:
- **X (predictors):** The original FM business metrics + One-Hot encoded categoricals
- **y (target):** The K-Means cluster label


In [ ]:
# Drop identifier
df_model = df.drop(columns=['CustomerID'])

# One-Hot encode categoricals
df_model = pd.get_dummies(df_model, columns=['Region', 'Preferred_Category'], drop_first=False)

# Convert booleans to int
for col in df_model.columns:
    if df_model[col].dtype == 'bool':
        df_model[col] = df_model[col].astype(int)

# Split features and target
y = df_model['Cluster']
X = df_model.drop(columns=['Cluster'])

feature_names = list(X.columns)
print(f"Features ({len(feature_names)}): {feature_names}")
print(f"Target classes: {sorted(y.unique())}")


### 6.3 Stratified Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} customers")
print(f"Test set:     {X_test.shape[0]} customers")


### 6.4 Decision Tree Training
We cap `max_depth=5` to keep the tree interpretable for academic reporting while still capturing the key splitting rules.


In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

dt_model.fit(X_train, y_train)
print("Decision Tree trained successfully.")
print(f"Tree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")


### 6.5 Model Evaluation


In [ ]:
y_pred = dt_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.2%}")
print(f"\n{'='*60}")
print("Classification Report:")
print('='*60)
print(classification_report(y_test, y_pred))


### 6.6 Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(y.unique()),
            yticklabels=sorted(y.unique()))
plt.xlabel('Predicted Cluster')
plt.ylabel('Actual Cluster')
plt.title('Decision Tree — Confusion Matrix')
plt.tight_layout()
plt.show()


### 6.7 Decision Tree Visualization
This is the core deliverable — a visual map of the splitting rules the model learned. Each node shows:
- The **feature** and **threshold** used to split
- The **Gini impurity** (lower = purer)
- The **number of samples** in that node
- The **predicted class** (cluster)


In [ ]:
plt.figure(figsize=(28, 14))
plot_tree(
    dt_model,
    feature_names=feature_names,
    class_names=[f"Cluster {i}" for i in sorted(y.unique())],
    filled=True,
    rounded=True,
    fontsize=9,
    proportion=True
)
plt.title('Decision Tree — Customer Segmentation Rules', fontsize=16)
plt.tight_layout()
plt.show()


### 6.8 Decision Rules (Text Export)
For the academic report, here are the exact splitting rules in plain text:


In [ ]:
rules = export_text(dt_model, feature_names=feature_names)
print(rules)


### 6.9 Feature Importance Ranking
Which features matter most in determining cluster membership? This directly answers the business question: **what drives customer segmentation?**


In [ ]:
importances = pd.Series(dt_model.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=True)

# Filter out zero-importance features for clarity
importances = importances[importances > 0]

plt.figure(figsize=(10, max(6, len(importances) * 0.4)))
importances.plot(kind='barh', color='teal')
plt.xlabel('Feature Importance (Gini)')
plt.title('Decision Tree — Feature Importance for Cluster Prediction')
plt.tight_layout()
plt.show()


### 6.10 Business Interpretation & Conclusions

**Key Takeaways from the Decision Tree:**

1. **Interpretability:** The tree provides clear, actionable rules that a business stakeholder can understand without any statistical background. Each path from root to leaf describes a customer archetype.

2. **Feature Importance:** The top-ranked features reveal what truly differentiates customer segments — whether it's spending volume, purchase frequency, geographic region, or product category preference.

3. **Actionable Segments:** By combining the K-Means cluster profiles (Section 5.7) with the Decision Tree rules, marketing teams can:
   - Design targeted campaigns per segment
   - Identify high-value customers at risk of churn
   - Optimize product recommendations per behavioral group

4. **Model Validity:** The classification accuracy on the held-out test set validates that the K-Means segments are *not arbitrary* — they correspond to real, learnable behavioral patterns in the data.
